# A.3: Regression check on held-out text

Compare the perplexity of the base and instruction-tuned models on held-out Shakespeare.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 Verifiable instruction checks
import json, re, math

import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

models = {name: AutoModelForCausalLM.from_pretrained(f"HuggingFaceTB/{name}",
                                                     dtype=torch.float32).eval()
          for name in ["SmolLM2-135M", "SmolLM2-135M-Instruct"]}

def words(t): return re.findall(r"\b\w+\b", t)

def bullets(t): return [l for l in t.splitlines() if re.match(r"^\s*[-*]\s+", l)]

def is_json(t):
    try:
        json.loads(t.strip().strip("`").removeprefix("json").strip())
        return True
    except ValueError:
        return False

checks = [
    ("Describe the moon in fewer than 30 words.", lambda t: 0 < len(words(t)) < 30),
    ("Describe a cat in fewer than 20 words.", lambda t: 0 < len(words(t)) < 20),
    ("List exactly three fruits as a bulleted list using '-' for each bullet.",
     lambda t: len(bullets(t)) == 3),
    ("Give exactly two tips for sleeping well, as a bulleted list using '-'.",
     lambda t: len(bullets(t)) == 2),
    ("Write one sentence about rain, entirely in lowercase letters.",
     lambda t: len(words(t)) > 0 and t == t.lower()),
    ("Write one sentence about the sun, in capital letters only.",
     lambda t: len(words(t)) > 0 and t == t.upper()),
    ("Answer with only the word yes or no: is fire hot?",
     lambda t: t.strip().strip(".").lower() in {"yes", "no"}),
    ("Name a color. End your answer with the exact phrase 'That is all.'",
     lambda t: t.strip().endswith("That is all.")),
    ('Return a JSON object with the keys "name" and "age" for a person called Ana who is 30.',
     is_json),
    ("Write the word hello, and nothing else.", lambda t: t.strip().strip(".!").lower() == "hello"),
]

def answer(model, prompt, max_new_tokens=80):
    text = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                   add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").input_ids
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                             eos_token_id=tok.convert_tokens_to_ids("<|im_end|>"),
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

results = {}

for name, model in models.items():
    results[name] = [(p, answer(model, p)) for p, _ in checks]
    passed = [check(a) for (_, check), (_, a) in zip(checks, results[name])]
    print(f"{name}: {sum(passed)}/{len(checks)} checks passed ->",
          "".join("P" if ok else "." for ok in passed))


In [ ]:
import os, urllib.request
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("shakespeare.txt"):
    urllib.request.urlretrieve(URL, "shakespeare.txt")
text = open("shakespeare.txt", encoding="utf-8").read()
held = tok(text[-20000:], add_special_tokens=False).input_ids
x = torch.tensor([held[i:i + 128] for i in range(0, 20 * 128, 128)])
for name, model in models.items():
    with torch.no_grad():
        print(f"{name}: perplexity {math.exp(model(x, labels=x).loss.item()):.1f}")


**Expected output**

Output:

```text
SmolLM2-135M: perplexity 41.7
SmolLM2-135M-Instruct: perplexity 56.5
```
